In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths
TRAIN_IMAGES_PATH = '/Users/mac/Development/diploma/skin-mutation-detection/data/train/ISIC_2019_Training_Input'  # Update with the path to your training images
TEST_IMAGES_PATH = '/Users/mac/Development/diploma/skin-mutation-detection/ham10000/HAM10000_images_part_12'    # Update with the path to your test images
METADATA_CSV = '/Users/mac/Development/diploma/skin-mutation-detection/data/train/ISIC_2019_Training_GroundTruth_Updated.csv'       # Update with the path to your metadata CSV
TEST_METADATA_CSV = '/Users/mac/Development/diploma/skin-mutation-detection/ham10000/HAM10000_Test_GroundTruth_Updated.csv'  # Update with the path to your test metadata CSV
ISIC_NEW_PATH = '/Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo'

CLASS_MAP = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

print("Loading training metadata...")
df = pd.read_csv(METADATA_CSV)

assert 'image' in df.columns, "Training metadata CSV must contain 'image' column."
assert all(cls in df.columns for cls in CLASS_MAP), "Training metadata CSV must contain all expected class columns."

print("Processing training labels...")
df["label"] = df[CLASS_MAP].idxmax(axis=1)
df = df[["image", "label"]]

# Split dataset
train_df, val_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)

# Read test metadata
print("Loading test metadata...")
test_df = pd.read_csv(TEST_METADATA_CSV)

assert 'image' in test_df.columns, "Test metadata CSV must contain 'image' column."
assert all(cls in test_df.columns for cls in CLASS_MAP), "Test metadata CSV must contain all expected class columns."

test_df["label"] = test_df[CLASS_MAP].idxmax(axis=1)
test_df = test_df[["image", "label"]]

def copy_images(dataframe, split, image_path):
    for _, row in dataframe.iterrows():
        src_path = os.path.join(image_path, f"{row['image']}.jpg")
        dest_dir = os.path.join(ISIC_NEW_PATH, split, row['label'])
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = os.path.join(dest_dir, f"{row['image']}.jpg")
        if os.path.exists(src_path):
            shutil.copy(src_path, dest_path)
        else:
            print(f"Warning: {src_path} does not exist.")

print("Copying images...")
copy_images(train_df, 'train', TRAIN_IMAGES_PATH)
copy_images(val_df, 'val', TRAIN_IMAGES_PATH)
copy_images(test_df, 'test', TEST_IMAGES_PATH)

print("Dataset restructuring complete!")

Loading training metadata...
Processing training labels...
Loading test metadata...
Copying images...
Dataset restructuring complete!


In [4]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11n.pt")

train_results = model.train(
    data="/Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo", 
    epochs=100,  
    imgsz=456,  
    device="mps",  
    task="classify"
)

metrics = model.val()

results = model("path/to/image.jpg")
results[0].show()

path = model.export(format="onnx")

Ultralytics 8.3.86 🚀 Python-3.11.11 torch-2.6.0 MPS (Apple M1 Pro)
engine/trainer: task=classify, mode=train, model=yolo11n.pt, data=/Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=mps, workers=8, project=None, name=train6, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=T

train: Scanning /Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo/train/AK.cache... 0 images, 17731 backgrounds, 0 corrupt: 100%|██████████| 17731/17731 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo/train/AK.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.



val: Scanning /Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo/val/AK.cache... 0 images, 7600 backgrounds, 0 corrupt: 100%|██████████| 7600/7600 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /Users/mac/Development/diploma/skin-mutation-detection/data/isic-yolo/val/AK.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
Plotting labels to runs/classify/train6/labels.jpg... 
zero-size array to reduction operation maximum which has no identity
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)


TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/classify/train6
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      10.7G          0      20.09          0          0        640: 100%|██████████| 1109/1109 [23:38<00:00,  1.28s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 238/238 [03:39<00:00,  1.08it/s]


RuntimeError: torch.cat(): expected a non-empty list of Tensors